Goal: build and train a CNN using PyTorch's nn.Conv2d/pooling layers





Steps

Load CIFAR-10 via torchvision.datasets.CIFAR10, using the book's own normalization transform. Set up DataLoaders for train/test in batches (the book uses batch size 8 as an example — worth trying something more standard like 32 or 64 for real training).
Define a CNN architecture, following the book's example structure: a couple of Conv2d + ReLU + pooling blocks (mixing max-pool and avg-pool, as the book does, is a fine way to see both), flattened into fully-connected layers at the end (nn.Linear) for the final 10-class output. You can start from the book's exact architecture, then modify it (more filters, an extra conv layer, different kernel sizes) once the base version works.





Full training loop — the book only shows forward + backward on a single image as a teaching example. Your job is to extend that into a real loop: iterate over all batches in train_loader, for each: forward pass → compute loss (cross-entropy is more appropriate here than the book's squared-error example, since this is multi-class classification) → loss.backward() → optimizer.step() → optimizer.zero_grad(). Repeat for multiple epochs, tracking loss per epoch.





Evaluate on the test set — run the trained model over test_loader (no gradient updates), compute accuracy. Compare to a naive baseline (10% for random guessing across 10 classes) — is the CNN meaningfully better?

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
#transform.ToTensor converts image into a torch.Tensor and rescales pixel values down to 0.0-1.0 floats, transforms.Normalize(mean, std) takes a mean and std for each of the 3 color channels (R,G,B) and centers each of the inputs around 0
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
train_set = CIFAR10(root = './data', train=True, download=True, transform=transform)
test_set = CIFAR10(root = './data', train=False, download=True, transform=transform)
len(train_set)
img, label = train_set[0]
img.shape
label
train_set.classes
train_set.classes[label]
#train_set can only feed in one example at a time, but training is done in batches, DataLoader feeds the model 32 images at once
train_loader = DataLoader(train_set, batch_size = 32, shuffle=True)
test_loader = DataLoader(test_set, batch_size = 32, shuffle=False)
#establish FFNN baseline
#CIFAR-10 images come out of the DataLoader as [batch, 3, 32, 32] -> a batch of 3 channel 32x32 images, to execute FFNN need a flat 1D vector for the input dimensions, use nn.Flatten() -> [batch, 3*32*32] = [batch, 3072]
#use two hidden layers (3072 -> 512 -> 128 -> 10)
class FFNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(3072, 512),
        nn.ReLU(),
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128,10)
    )
  def forward(self, x):
    return self.model(x)
net = FFNN()
optimizer = optim.SGD(net.parameters(), lr = 0.01)
criterion = nn.CrossEntropyLoss()
num_epochs = 5
for epoch in range(num_epochs):
  running_loss = 0.0
  for images, labels in train_loader:
    optimizer.zero_grad() #clear old gradients
    outputs = net(images) #calls forward pass logic
    loss = criterion(outputs, labels) #computes loss, softmax process is folded under this
    loss.backward() #computes new gradients
    optimizer.step() #updates weights using those gradients
    running_loss += loss.item()
  print(f"Epoch {epoch+1}, avg loss: {running_loss/len(train_loader):.4f}")
net.eval()
correct = 0
total = 0
with torch.no_grad():
  for images, labels in test_loader:
    outputs = net(images)
    predicted = torch.argmax(outputs, dim=1)
    correct += (predicted == labels).sum().item()
    total += labels.size(0)
print(f"Test Accuracy: {correct/total:.4f}")

100%|██████████| 170M/170M [1:00:23<00:00, 47.1kB/s]


Epoch 1, avg loss: 1.9029
Epoch 2, avg loss: 1.6119
Epoch 3, avg loss: 1.4995
Epoch 4, avg loss: 1.4139
Epoch 5, avg loss: 1.3455
Test Accuracy: 0.5042


In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
#input has 3 channels (R, G, B), designed to have 16 filters, while the size  of each filter is 5x5 (kernel, looks at small matrices rather than just individual pixels to indentify patterns)
#nn.Conv2d instantiates an object - creates 16 filters, each size 3x5x5, filter values are randomly intialized
    nn.Conv2d(3, 16, 5, padding = 2),
    nn.ReLU(),
#max pooling reduces spatial size while keeping the important information(after Conv2d detects patterns, get a full feature map, but don't need a full map to know whether a pattern was detected in a general area, and max pooling throws away the exact position)
    nn.MaxPool2d(2, stride = None),
#devlop more layers - early layers detect simple, low-level patterns, later layers combine these into more complex, abstract patterns(FFNN starts with every raw pixel and tries to compress towards the answer, whereas here the goal is to build towards a more complex system of pattern tracking)
    nn.Conv2d(16, 32, 5, padding = 2),
    nn.ReLU(),
    nn.MaxPool2d(2, stride = None),
    nn.Conv2d(32, 64, 5, padding = 2),
    nn.ReLU(),
    nn.MaxPool2d(2, stride = None),
#want the model to classify each input as 1 of 10 possible outputs, so need to map the final vector down to a linear output of size 10
    nn.Flatten(),
#if (2xpadding) - kernel_size +1 = 0, then the size is unchanged from Conv2d, whereas MaxPool2d(2) halves the spatial size, meaning that final_size = input_size/(2^n) with n = # of pooling layers, input of 32x32, 3 pooling layers -> 32/(2^3) -> 4x4, multiply this by final # of filters (64) -> 1024 = 4x4x64
    nn.Linear(1024, 10)
    )
  def forward(self, x):
    return self.model(x)
net_CNN = CNN()
optimizer_CNN = optim.SGD(net_CNN.parameters(), lr = 0.01)
criterion_CNN = nn.CrossEntropyLoss()
num_epochs = 10
for i in range(num_epochs):
  for images, labels in train_loader:
    optimizer_CNN.zero_grad()
    output_CNN = net_CNN(images)
    loss_CNN = criterion_CNN(output_CNN, labels)
    loss_CNN.backward()
    optimizer_CNN.step()
net_CNN.eval()
correct_CNN = 0
total_CNN = 0
for images, labels in test_loader:
  with torch.no_grad():
    correct_CNN += (torch.argmax(net_CNN(images), dim=1)==labels).sum().item()
    total_CNN += len(labels)
CNN_accuracy = correct_CNN/total_CNN
print(CNN_accuracy)

0.7019
